# Final Feature Engineering

In [1]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

# Load preprocessed train and test data
X_train_final = pd.read_parquet('../data/cleaned/X_train_preprocessed.parquet')
X_test_final = pd.read_parquet('../data/cleaned/X_test_preprocessed.parquet')
y_train = pd.read_parquet('../data/cleaned/y_train.parquet').squeeze()
y_test = pd.read_parquet('../data/cleaned/y_test.parquet').squeeze()

print("="*70)
print("DATA LOADED SUCCESSFULLY")
print("="*70)
print(f"✓ X_train shape: {X_train_final.shape}")
print(f"✓ X_test shape: {X_test_final.shape}")
print(f"✓ y_train shape: {y_train.shape}")
print(f"✓ y_test shape: {y_test.shape}")

DATA LOADED SUCCESSFULLY
✓ X_train shape: (105905, 40)
✓ X_test shape: (26477, 40)
✓ y_train shape: (105905,)
✓ y_test shape: (26477,)


In [2]:
# Read data from proms_providers_academic_categories.xlsx

proms_academic_df = pd.read_excel('../data/supporting/proms_providers_academic_categories.xlsx')

proms_academic_df.head()

# Transform academisch/niet academisch to 1/0
proms_academic_df['University Hospital'] = proms_academic_df['category'].map({'academisch': 1, 'niet-academisch': 0})
proms_academic_df.head(50)

# Merge with X_train_final and X_test_final
X_train_final = X_train_final.merge(proms_academic_df[['provider_code', 'University Hospital']], on='provider_code', how='left')
X_test_final = X_test_final.merge(proms_academic_df[['provider_code', 'University Hospital']], on='provider_code', how='left')

X_train_final['University Hospital'].value_counts()
X_test_final['University Hospital'].value_counts()

University Hospital
0    20694
1     5783
Name: count, dtype: int64

In [3]:
# Read data from proms_providers_academic_categories.xlsx

proms_hospital_info = pd.read_excel('../data/supporting/ODS Advanced Search 2.xlsx')

# Example - will customize based on your actual values
proms_hospital_info['independent_hospital'] = proms_hospital_info['Primary Role Name'].map({
    'INDEPENDENT SECTOR H/C PROVIDER SITE': 1,
    'NHS TRUST': 0,
    'INDEPENDENT SECTOR HEALTHCARE PROVIDER': 1
})

proms_hospital_info['provider_code'] = proms_hospital_info['Dataset']
# Merge with X_train_final and X_test_final
X_train_final = X_train_final.merge(proms_hospital_info[['provider_code', 'independent_hospital', 'Region']], on='provider_code', how='left')
X_test_final = X_test_final.merge(proms_hospital_info[['provider_code', 'independent_hospital', 'Region']], on='provider_code', how='left')

X_train_final['Region'].value_counts()
proms_hospital_info['Primary Role Name'].value_counts()

Primary Role Name
INDEPENDENT SECTOR H/C PROVIDER SITE      156
NHS TRUST                                 136
INDEPENDENT SECTOR HEALTHCARE PROVIDER      2
Name: count, dtype: int64

In [4]:
# Extract top 10 most common provider codes with their value counts and how much they appear in the training set (% wise of total)
top_50_providers = X_train_final['provider_code'].value_counts().nsmallest(50).index.tolist()

for provider in top_50_providers:
    count = X_train_final['provider_code'].value_counts()[provider]
    pct = (count / len(X_train_final)) * 100
    print(f"Provider Code: {provider}, Count: {count}, Percentage of Training Set: {pct:.2f}%")
# ============================================================

Provider Code: RJ7, Count: 5, Percentage of Training Set: 0.00%
Provider Code: NT30A, Count: 7, Percentage of Training Set: 0.01%
Provider Code: NYW01, Count: 8, Percentage of Training Set: 0.01%
Provider Code: NT310, Count: 9, Percentage of Training Set: 0.01%
Provider Code: NT343, Count: 9, Percentage of Training Set: 0.01%
Provider Code: NT455, Count: 10, Percentage of Training Set: 0.01%
Provider Code: NFH01, Count: 11, Percentage of Training Set: 0.01%
Provider Code: NT315, Count: 12, Percentage of Training Set: 0.01%
Provider Code: NT416, Count: 16, Percentage of Training Set: 0.02%
Provider Code: NVC0R, Count: 16, Percentage of Training Set: 0.02%
Provider Code: NQM01, Count: 17, Percentage of Training Set: 0.02%
Provider Code: NT308, Count: 20, Percentage of Training Set: 0.02%
Provider Code: NT241, Count: 22, Percentage of Training Set: 0.02%
Provider Code: NT436, Count: 22, Percentage of Training Set: 0.02%
Provider Code: NT211, Count: 23, Percentage of Training Set: 0.02%
Pr

### Comorbidity counter

In [5]:
# Create a new variable, comorbidity counter based on all conditions

health_conditions = [
    'heart_disease', 'high_bp', 'stroke', 'circulation',
    'lung_disease', 'diabetes', 'kidney_disease',
    'nervous_system', 'liver_disease', 'cancer',
    'depression', 'arthritis'
]
X_train_final['comorbidity_count'] = X_train_final[health_conditions].sum(axis=1)
X_test_final['comorbidity_count'] = X_test_final[health_conditions].sum(axis=1)

X_train_final['comorbidity_count'].value_counts().sort_index()

comorbidity_count
0     10920
1     36181
2     34655
3     16688
4      5460
5      1453
6       382
7        68
8        18
9         7
10       14
11       18
12       41
Name: count, dtype: int64

In [6]:
X_train_final.columns.tolist()

['provider_code',
 'year',
 'age_band',
 'gender',
 't0_assisted',
 't0_assisted_by',
 't0_symptom_period',
 't0_previous_surgery',
 't0_living_arrangements',
 't0_disability',
 'heart_disease',
 'high_bp',
 'stroke',
 'circulation',
 'lung_disease',
 'diabetes',
 'kidney_disease',
 'nervous_system',
 'liver_disease',
 'cancer',
 'depression',
 'arthritis',
 't0_mobility',
 't0_self_care',
 't0_activity',
 't0_discomfort',
 't0_anxiety',
 'oks_t0_pain',
 'oks_t0_night_pain',
 'oks_t0_washing',
 'oks_t0_transport',
 'oks_t0_walking',
 'oks_t0_standing',
 'oks_t0_limping',
 'oks_t0_kneeling',
 'oks_t0_work',
 'oks_t0_confidence',
 'oks_t0_shopping',
 'oks_t0_stairs',
 'oks_t0_score',
 'University Hospital',
 'independent_hospital',
 'Region',
 'comorbidity_count']

### OKS Subscales

In [8]:
# Create new variables, oks subscales for pain, function and activities of daily life

X_train_final['oks_pain_subscale'] = (
    X_train_final['oks_t0_pain'] + X_train_final['oks_t0_night_pain']
)
X_train_final['oks_function_subscale'] = (
    X_train_final['oks_t0_washing']  + X_train_final['oks_t0_stairs'] +
    X_train_final['oks_t0_walking'] + X_train_final['oks_t0_standing'] +
    X_train_final['oks_t0_limping'] + X_train_final['oks_t0_kneeling']
)
X_train_final['oks_adl_subscale'] = (
    X_train_final['oks_t0_work'] + X_train_final['oks_t0_confidence'] +
    X_train_final['oks_t0_shopping'] + X_train_final['oks_t0_transport']
)

# Apply same for test set

X_test_final['oks_pain_subscale'] = (
    X_test_final['oks_t0_pain'] + X_test_final['oks_t0_night_pain']
)

X_test_final['oks_function_subscale'] = (
    X_test_final['oks_t0_washing']  + X_test_final['oks_t0_stairs'] +
    X_test_final['oks_t0_walking'] + X_test_final['oks_t0_standing'] +
    X_test_final['oks_t0_limping'] + X_test_final['oks_t0_kneeling']
)

X_test_final['oks_adl_subscale'] = (
    X_test_final['oks_t0_work'] + X_test_final['oks_t0_confidence'] +
    X_test_final['oks_t0_shopping'] + X_test_final['oks_t0_transport']
)
X_train_final[['oks_pain_subscale', 'oks_function_subscale', 'oks_adl_subscale']].head()

,oks_pain_subscale,oks_function_subscale,oks_adl_subscale
0,2,10,8
1,0,12,11
2,0,8,5
3,2,10,10
4,3,14,10


In [9]:
# ============================================================
# ONE-HOT ENCODE CATEGORICAL VARIABLES
# ============================================================

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import joblib

# Define categorical columns to encode
categorical_columns = [
    'year', 'age_band', 
    't0_living_arrangements', 'Region'
]

print("="*70)
print("ONE-HOT ENCODING CATEGORICAL VARIABLES")
print("="*70)

# Check which categorical columns exist in the data
existing_categorical = [col for col in categorical_columns if col in X_train_final.columns]
print(f"\nCategorical columns found: {existing_categorical}")

# ============================================================
# FIT ONE-HOT ENCODER ON TRAINING DATA
# ============================================================

# Initialize encoder
encoder = OneHotEncoder(
    drop='first',           # Drop first category to avoid multicollinearity
    sparse_output=False,    # Return dense array (easier to work with)
    handle_unknown='ignore' # Ignore unknown categories in test set
)

# Fit on training data
print(f"\n✓ Fitting encoder on training data...")
encoder.fit(X_train_final[existing_categorical])

# Get feature names for encoded columns
encoded_feature_names = encoder.get_feature_names_out(existing_categorical)
print(f"✓ Encoder fitted - will create {len(encoded_feature_names)} one-hot encoded features")

# ============================================================
# TRANSFORM TRAINING DATA
# ============================================================

print(f"\n✓ Transforming training data...")

# Transform categorical columns
X_train_encoded = encoder.transform(X_train_final[existing_categorical])

# Create DataFrame with encoded features
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train_final.index
)

# Drop original categorical columns and add encoded ones
X_train_numerical = X_train_final.drop(columns=existing_categorical)
X_train_final_encoded = pd.concat([X_train_numerical, X_train_encoded_df], axis=1)

print(f"✓ Training data shape before encoding: {X_train_final.shape}")
print(f"✓ Training data shape after encoding: {X_train_final_encoded.shape}")

# ============================================================
# TRANSFORM TEST DATA
# ============================================================

print(f"\n✓ Transforming test data...")

# Transform categorical columns (using fitted encoder)
X_test_encoded = encoder.transform(X_test_final[existing_categorical])

# Create DataFrame with encoded features
X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test_final.index
)

# Drop original categorical columns and add encoded ones
X_test_numerical = X_test_final.drop(columns=existing_categorical)
X_test_final_encoded = pd.concat([X_test_numerical, X_test_encoded_df], axis=1)

print(f"✓ Test data shape before encoding: {X_test_final.shape}")
print(f"✓ Test data shape after encoding: {X_test_final_encoded.shape}")

# ============================================================
# VERIFICATION
# ============================================================

print("\n" + "="*70)
print("ENCODING VERIFICATION")
print("="*70)

print(f"\n✓ X_train_final_encoded shape: {X_train_final_encoded.shape}")
print(f"✓ X_test_final_encoded shape: {X_test_final_encoded.shape}")
print(f"✓ Both datasets have same number of features: {X_train_final_encoded.shape[1] == X_test_final_encoded.shape[1]}")

# Check for missing values
print(f"\n✓ Missing values in X_train_final_encoded: {X_train_final_encoded.isna().sum().sum()}")
print(f"✓ Missing values in X_test_final_encoded: {X_test_final_encoded.isna().sum().sum()}")

# Show sample of new columns
print("\n" + "="*70)
print("SAMPLE OF ONE-HOT ENCODED FEATURES")
print("="*70)

# Show first few encoded feature names for each original categorical variable
for original_col in existing_categorical:
    encoded_cols = [col for col in encoded_feature_names if col.startswith(original_col)]
    print(f"\n{original_col} → {len(encoded_cols)} one-hot features:")
    print(f"  Sample: {encoded_cols[:5]}")

# Show data sample
print("\n" + "="*70)
print("DATA PREVIEW (First 5 rows, first 10 columns)")
print("="*70)
print(X_train_final_encoded.iloc[:5, :10])

# ============================================================
# SAVE ENCODED DATA (Optional)
# ============================================================

print("\n" + "="*70)
print("SAVING ENCODED DATA")
print("="*70)

# Drop provider_code before saving

X_train_final_encoded = X_train_final_encoded.drop(columns=['provider_code'])
X_test_final_encoded = X_test_final_encoded.drop(columns=['provider_code'])

# Save encoded datasets
X_train_final_encoded.to_parquet('../data/cleaned/X_train_encoded_wo_provider.parquet', index=False)
X_test_final_encoded.to_parquet('../data/cleaned/X_test_encoded_wo_provider.parquet', index=False)

# Save encoder for future use
joblib.dump(encoder, '../data/cleaned/onehot_encoder_wo_provider.pkl')

print("✓ X_train_final_encoded saved to '../data/cleaned/X_train_encoded.parquet'")
print("✓ X_test_final_encoded saved to '../data/cleaned/X_test_encoded.parquet'")
print("✓ OneHotEncoder saved to '../data/cleaned/onehot_encoder.pkl'")

# ============================================================
# FEATURE COUNTS BY TYPE
# ============================================================

print("\n" + "="*70)
print("FEATURE BREAKDOWN")
print("="*70)

# Count features by original type
numerical_features = X_train_numerical.columns.tolist()
encoded_features = encoded_feature_names.tolist()

print(f"\nNumerical features (unchanged): {len(numerical_features)}")
print(f"One-hot encoded features (new): {len(encoded_features)}")
print(f"Total features: {len(X_train_final_encoded.columns)}")

print("\n" + "="*70)
print("✓ ONE-HOT ENCODING COMPLETE - DATA READY FOR MODELING")
print("="*70)

# ============================================================
# How to load encoder and apply to new data later:
# ============================================================
"""
# Load the fitted encoder
encoder = joblib.load('../data/processed/onehot_encoder.pkl')

# Apply to new data
X_new_encoded = encoder.transform(X_new[existing_categorical])
X_new_encoded_df = pd.DataFrame(
    X_new_encoded,
    columns=encoder.get_feature_names_out(existing_categorical),
    index=X_new.index
)
X_new_numerical = X_new.drop(columns=existing_categorical)
X_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)
"""

ONE-HOT ENCODING CATEGORICAL VARIABLES

Categorical columns found: ['year', 'age_band', 't0_living_arrangements', 'Region']

✓ Fitting encoder on training data...
✓ Encoder fitted - will create 15 one-hot encoded features

✓ Transforming training data...
✓ Training data shape before encoding: (105905, 47)
✓ Training data shape after encoding: (105905, 58)

✓ Transforming test data...
✓ Test data shape before encoding: (26477, 47)
✓ Test data shape after encoding: (26477, 58)

ENCODING VERIFICATION

✓ X_train_final_encoded shape: (105905, 58)
✓ X_test_final_encoded shape: (26477, 58)
✓ Both datasets have same number of features: True

✓ Missing values in X_train_final_encoded: 0
✓ Missing values in X_test_final_encoded: 0

SAMPLE OF ONE-HOT ENCODED FEATURES

year → 2 one-hot features:
  Sample: ['year_2017/18', 'year_2018/19']

age_band → 3 one-hot features:
  Sample: ['age_band_3', 'age_band_4', 'age_band_5']

t0_living_arrangements → 2 one-hot features:
  Sample: ['t0_living_arrangeme

"\n# Load the fitted encoder\nencoder = joblib.load('../data/processed/onehot_encoder.pkl')\n\n# Apply to new data\nX_new_encoded = encoder.transform(X_new[existing_categorical])\nX_new_encoded_df = pd.DataFrame(\n    X_new_encoded,\n    columns=encoder.get_feature_names_out(existing_categorical),\n    index=X_new.index\n)\nX_new_numerical = X_new.drop(columns=existing_categorical)\nX_new_final = pd.concat([X_new_numerical, X_new_encoded_df], axis=1)\n"